# 05 Pure Quantum Models Training

**No CNN backbone — pure quantum circuits on handcrafted features**

Three quantum approaches compared:
1. **Pure VQC** — AngleEmbedding + StronglyEntanglingLayers
2. **Quantum Kernel SVM (QSVM)** — Quantum kernel matrix + classical SVC
3. **Data Re-uploading QNN** — Input re-encoded at every layer

All models operate on the 6-dimensional handcrafted features.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from utils.config import CFG
from data.loader import load_dataset
from data.preprocess import build_feature_matrix

print('Loading dataset (200 images per class)...')
train_rec, val_rec, test_rec = load_dataset(max_per_class=200)

print('Extracting 6-dim handcrafted features...')
X_train, y_train_raw = build_feature_matrix(train_rec, desc='Train')
X_val, y_val_raw = build_feature_matrix(val_rec, desc='Val')
X_test, y_test_raw = build_feature_matrix(test_rec, desc='Test')

# Remap labels to contiguous 0..n_classes-1
le = LabelEncoder()
le.fit(np.concatenate([y_train_raw, y_val_raw, y_test_raw]))
y_train = le.transform(y_train_raw)
y_val = le.transform(y_val_raw)
y_test = le.transform(y_test_raw)

# Normalize features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

n_features = X_train.shape[1]
n_classes = len(le.classes_)
print(f'Features: {n_features}, Classes: {n_classes}')
print(f'Labels remapped to 0..{n_classes-1}')
print(f'Shapes -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

C:\Users\balaj\AppData\Local\Programs\Python\Python38\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: Could not find module 'C:\Users\balaj\AppData\Local\Programs\Python\Python38\Lib\site-packages\torchvision\image.pyd' (or one of its dependencies). Try using the full path with constructor syntax.
  warn(f"Failed to load image Python extension: {e}")


Loading dataset (200 images per class)...
  Auto-discovered 101 classes in MasterDataset/


Dataset loaded → train: 11007, val: 1572, test: 3144
  Train class distribution:
    [0] american_bollworm_on_cotton    → 63 images
    [1] anthracnose_on_cotton          → 20 images
    [2] apple_apple_scab               → 142 images
    [3] apple_black_rot                → 151 images
    [4] apple_cedar_apple_rust         → 145 images
    [5] apple_healthy                  → 131 images
    [6] army_worm                      → 148 images
    [7] bacterial_blight_in_cotton     → 147 images
    [8] becterial_blight_in_rice       → 137 images
    [9] bell_pepper_bacterial_spot     → 146 images
    [10] bell_pepper_healthy            → 122 images
    [11] blueberry_healthy              → 153 images
    [12] bollrot_on_cotton              → 2 images
    [14] brownspot                      → 148 images
    [15] cherry_healthy                 → 142 images
    [16] cherry_including_sour_healthy  → 144 images
    [17] cherry_including_sour_powdery_mildew → 145 images
    [18] cherry_powdery_mi

Train:   0%| | 0/11007 [00:00<?,

Train:   1%| | 132/11007 [00:00<

...

Test:  66%|▋| 2062/3144 [00:04<0

Test:  77%|▊| 2421/3144 [00:04<0

Test:  86%|▊| 2698/3144 [00:05<0

Test: 100%|█| 3144/3144 [00:05<0

Features: 6, Classes: 92
Labels remapped to 0..91
Shapes -> Train: (11007, 6), Val: (1572, 6), Test: (3144, 6)


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MODEL 1: Pure VQC (AngleEmbedding + StronglyEntanglingLayers)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('='*60)
print('MODEL 1: Pure VQC')
print('  AngleEmbedding + StronglyEntanglingLayers (6q, 6L)')
print('='*60)

n_q = n_features
n_l = 6
dev1 = qml.device('default.qubit', wires=n_q)

@qml.qnode(dev1, interface='torch', diff_method='backprop')
def pure_vqc_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_q), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(n_q))
    return [qml.expval(qml.PauliZ(w)) for w in range(n_q)]

class PureVQC(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(pure_vqc_circuit, {'weights': (n_l, n_q, 3)})
        self.classifier = nn.Sequential(nn.Linear(n_q, 32), nn.ReLU(), nn.Linear(32, n_classes))
    def forward(self, x):
        return self.classifier(self.qlayer(x))

pure_vqc = PureVQC()
opt1 = torch.optim.Adam(pure_vqc.parameters(), lr=0.01)
crit = nn.CrossEntropyLoss()
X_tr = torch.tensor(X_train_s, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_te = torch.tensor(X_test_s, dtype=torch.float32)

t0 = time.time()
for epoch in range(1, 31):
    pure_vqc.train()
    perm = torch.randperm(len(X_tr))
    eloss = 0.0
    for i in range(0, len(X_tr), 64):
        idx = perm[i:i+64]
        opt1.zero_grad()
        loss = crit(pure_vqc(X_tr[idx]), y_tr[idx])
        loss.backward()
        opt1.step()
        eloss += loss.item()
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:>3}/30 | Loss: {eloss/(len(X_tr)//64):.4f} | {time.time()-t0:.0f}s')

pure_vqc.eval()
with torch.no_grad():
    preds_vqc = torch.argmax(pure_vqc(X_te), dim=1).numpy()
vqc_acc = accuracy_score(y_test, preds_vqc)
vqc_f1 = f1_score(y_test, preds_vqc, average='macro', zero_division=0)
print(f'\nPure VQC -> Accuracy: {vqc_acc:.4f}, Macro F1: {vqc_f1:.4f} ({time.time()-t0:.0f}s)')

MODEL 1: Pure VQC
  AngleEmbedding + StronglyEntanglingLayers (6q, 6L)


  Epoch   1/30 | Loss: 4.0264 | 22s


  Epoch   5/30 | Loss: 3.2384 | 286s


  Epoch  10/30 | Loss: 3.0770 | 535s


  Epoch  15/30 | Loss: 3.0070 | 642s


  Epoch  20/30 | Loss: 2.9637 | 749s


  Epoch  25/30 | Loss: 2.9319 | 854s


  Epoch  30/30 | Loss: 2.9121 | 960s

Pure VQC -> Accuracy: 0.2172, Macro F1: 0.1647 (960s)


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MODEL 2: Quantum Kernel SVM (QSVM)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('='*60)
print('MODEL 2: Quantum Kernel SVM (QSVM)')
print('  ZZ Feature Map kernel + classical SVC')
print('='*60)

dev2 = qml.device('default.qubit', wires=n_features)

@qml.qnode(dev2, interface='numpy')
def qkernel_circuit(x1, x2):
    qml.AngleEmbedding(x1, wires=range(n_features), rotation='Y')
    for i in range(n_features - 1):
        qml.CNOT(wires=[i, i+1])
        qml.RZ(x1[i] * x1[i+1], wires=i+1)
        qml.CNOT(wires=[i, i+1])
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_features), rotation='Y')
    for i in range(n_features - 1):
        qml.CNOT(wires=[i, i+1])
        qml.RZ(-x2[i] * x2[i+1], wires=i+1)
        qml.CNOT(wires=[i, i+1])
    return qml.probs(wires=range(n_features))

def qkernel(x1, x2):
    return qkernel_circuit(x1, x2)[0]

# Subsample for kernel computation
np.random.seed(42)
N_TR = min(300, len(X_train_s))
N_TE = min(150, len(X_test_s))
idx_tr = np.random.choice(len(X_train_s), N_TR, replace=False)
idx_te = np.random.choice(len(X_test_s), N_TE, replace=False)
Xk_tr, yk_tr = X_train_s[idx_tr], y_train[idx_tr]
Xk_te, yk_te = X_test_s[idx_te], y_test[idx_te]

print(f'Computing quantum kernel ({N_TR}x{N_TR} train, {N_TE}x{N_TR} test)...')
t0 = time.time()
K_train = np.zeros((N_TR, N_TR))
for i in range(N_TR):
    for j in range(i, N_TR):
        k = qkernel(Xk_tr[i], Xk_tr[j])
        K_train[i,j] = K_train[j,i] = k
    if (i+1) % 50 == 0:
        print(f'  Train kernel row {i+1}/{N_TR} ({time.time()-t0:.0f}s)')

K_test = np.zeros((N_TE, N_TR))
for i in range(N_TE):
    for j in range(N_TR):
        K_test[i,j] = qkernel(Xk_te[i], Xk_tr[j])
print(f'Kernel computation: {time.time()-t0:.0f}s')

qsvm = SVC(kernel='precomputed', C=1.0)
qsvm.fit(K_train, yk_tr)
preds_qsvm = qsvm.predict(K_test)
qsvm_acc = accuracy_score(yk_te, preds_qsvm)
qsvm_f1 = f1_score(yk_te, preds_qsvm, average='macro', zero_division=0)
print(f'\nQSVM -> Accuracy: {qsvm_acc:.4f}, Macro F1: {qsvm_f1:.4f}')

MODEL 2: Quantum Kernel SVM (QSVM)
  ZZ Feature Map kernel + classical SVC
Computing quantum kernel (300x300 train, 150x300 test)...


  Train kernel row 50/300 (135s)


  Train kernel row 100/300 (258s)


  Train kernel row 150/300 (333s)


  Train kernel row 200/300 (382s)


  Train kernel row 250/300 (412s)


  Train kernel row 300/300 (422s)


Kernel computation: 799s

QSVM -> Accuracy: 0.0400, Macro F1: 0.0292


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MODEL 3: Data Re-uploading QNN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('='*60)
print('MODEL 3: Data Re-uploading QNN (8 layers)')
print('='*60)

n_l3 = 8
dev3 = qml.device('default.qubit', wires=n_features)

@qml.qnode(dev3, interface='torch', diff_method='backprop')
def reupload_circuit(inputs, weights):
    for layer in range(n_l3):
        qml.AngleEmbedding(inputs, wires=range(n_features), rotation='Y')
        for q in range(n_features):
            qml.Rot(weights[layer,q,0], weights[layer,q,1], weights[layer,q,2], wires=q)
        for q in range(n_features):
            qml.CNOT(wires=[q, (q+1)%n_features])
        if n_features >= 4:
            for q in range(0, n_features-2, 2):
                qml.CNOT(wires=[q, q+2])
    return [qml.expval(qml.PauliZ(w)) for w in range(n_features)]

class DataReupQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(reupload_circuit, {'weights': (n_l3, n_features, 3)})
        self.classifier = nn.Sequential(nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n_classes))
    def forward(self, x):
        return self.classifier(self.qlayer(x))

qnn = DataReupQNN()
opt3 = torch.optim.Adam(qnn.parameters(), lr=0.005)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=50)

t0 = time.time()
best_acc = 0
for epoch in range(1, 51):
    qnn.train()
    perm = torch.randperm(len(X_tr))
    eloss = 0.0
    for i in range(0, len(X_tr), 64):
        idx = perm[i:i+64]
        opt3.zero_grad()
        loss = crit(qnn(X_tr[idx]), y_tr[idx])
        loss.backward()
        opt3.step()
        eloss += loss.item()
    sched.step()
    if epoch % 10 == 0 or epoch == 1:
        qnn.eval()
        with torch.no_grad():
            vp = torch.argmax(qnn(torch.tensor(X_val_s, dtype=torch.float32)), dim=1).numpy()
        va = accuracy_score(y_val, vp)
        if va > best_acc:
            best_acc = va
            torch.save(qnn.state_dict(), '../outputs/models/qnn_best.pt')
        print(f'  Epoch {epoch:>3}/50 | Loss: {eloss/(len(X_tr)//64):.4f} | Val: {va:.4f} | {time.time()-t0:.0f}s')

qnn.load_state_dict(torch.load('../outputs/models/qnn_best.pt'))
qnn.eval()
with torch.no_grad():
    preds_qnn = torch.argmax(qnn(X_te), dim=1).numpy()
qnn_acc = accuracy_score(y_test, preds_qnn)
qnn_f1 = f1_score(y_test, preds_qnn, average='macro', zero_division=0)
print(f'\nQNN -> Accuracy: {qnn_acc:.4f}, Macro F1: {qnn_f1:.4f} ({time.time()-t0:.0f}s)')

MODEL 3: Data Re-uploading QNN (8 layers)


  Epoch   1/50 | Loss: 4.3599 | Val: 0.0706 | 36s


  Epoch  10/50 | Loss: 3.6595 | Val: 0.1221 | 367s


  Epoch  20/50 | Loss: 3.5549 | Val: 0.1349 | 738s


  Epoch  30/50 | Loss: 3.4965 | Val: 0.1469 | 1098s


  Epoch  40/50 | Loss: 3.4847 | Val: 0.1476 | 2003s


  Epoch  50/50 | Loss: 3.4620 | Val: 0.1476 | 2363s



QNN -> Accuracy: 0.1326, Macro F1: 0.0913 (2363s)


In [5]:
print('\n' + '='*70)
print('  PURE QUANTUM MODELS — FINAL BENCHMARK')
print('='*70)
print(f'{"Model":<30} {"Accuracy":>12} {"Macro F1":>12}')
print('-'*70)
print(f'{"Pure VQC":<30} {vqc_acc:>12.4f} {vqc_f1:>12.4f}')
print(f'{"QSVM (Quantum Kernel)":<30} {qsvm_acc:>12.4f} {qsvm_f1:>12.4f}')
print(f'{"Data Re-uploading QNN":<30} {qnn_acc:>12.4f} {qnn_f1:>12.4f}')
print('-'*70)
print('Classical baselines (notebook 02, full dataset):')
print(f'{"  Random Forest":<30} {0.5793:>12.4f} {0.4477:>12.4f}')
print(f'{"  SVM (Linear)":<30} {0.2151:>12.4f} {0.0772:>12.4f}')
print(f'{"  Logistic Regression":<30} {0.2458:>12.4f} {0.1221:>12.4f}')
print('='*70)
best_q = max([(vqc_acc,'Pure VQC'),(qsvm_acc,'QSVM'),(qnn_acc,'Data Re-uploading QNN')])
print(f'\n Best Pure Quantum Model: {best_q[1]} with {best_q[0]:.4f} accuracy')
print('\n Done!')


  PURE QUANTUM MODELS — FINAL BENCHMARK
Model                              Accuracy     Macro F1
----------------------------------------------------------------------
Pure VQC                             0.2172       0.1647
QSVM (Quantum Kernel)                0.0400       0.0292
Data Re-uploading QNN                0.1326       0.0913
----------------------------------------------------------------------
Classical baselines (notebook 02, full dataset):
  Random Forest                      0.5793       0.4477
  SVM (Linear)                       0.2151       0.0772
  Logistic Regression                0.2458       0.1221

 Best Pure Quantum Model: Pure VQC with 0.2172 accuracy

 Done!
